# 03 – Vektorisierung

Hier werden die Wortlisten aus der Vorverarbeitung in Zahlen umgewandelt, damit die Themenmodelle damit rechnen können.

**Eingabe:** 
`data/tokens_final.pkl` (19.807 Beschwerden als Wortlisten)

**Ausgabe:** 
zwei Matrizen (Bag-of-Words und TF-IDF)

In [2]:
import pandas as pd
from collections import Counter

In [3]:
df = pd.read_pickle("data/tokens_final.pkl")
print(df.shape)
print(df.columns.tolist())

(19807, 6)
['Complaint ID', 'Product', 'Issue', 'text_clean', 'tokens_filtered', 'n_tokens']


In [4]:
print(type(df["tokens_filtered"].iloc[0]))     # sollte <class 'list'> sein
print(df["tokens_filtered"].iloc[0])           # Beispiel einer Wortliste
print(df["n_tokens"].describe())

<class 'list'>
['live', 'city', 'offer', 'tax', 'incentive', 'homeowner', 'certain', 'income', 'qualify', 'homeowner', 'credit', 'tax', 'credit', 'special', 'credit', 'income', 'impound', 'account', 'mortgage', 'company', 'cover', 'tax', 'mortgage', 'company', 'understand', 'program', 'work', 'apply', 'money', 'account', 'tax', 'pay', 'exception', 'mortgage', 'company', 'continue', 'pay', 'money', 'impound', 'account', 'year', 'fall', 'payment', 'unemployment', 'mortgage', 'company', 'begin', 'foreclosure', 'proceeding', 'tell', 'money', 'impound', 'account', 'tax', 'pay', 'catch', 'payment', 'realize', 'tax', 'pay', 'question', 'representative', 'rude', 'tell', 'contact', 'agency', 'begin', 'cooperate', 'tax', 'pay', 'overage', 'account', 'ask', 'receive', 'tell', 'time', 'analysis', 'account', 'current', 'pay', 'time', 'account', 'current', 'pay', 'representative', 'indicate', 'time', 'analysis', 'current', 'hold', 'tax', 'little', 'tax', 'credit', 'pay', 'balance', 'mortgage', 'comp

In [5]:
# bag of words test

from sklearn.feature_extraction.text import CountVectorizer

mini = [
    ["credit", "report", "error"],
    ["credit", "card", "fee", "fee"],
    ["report", "error", "error"],
]

cv_mini = CountVectorizer(analyzer=lambda tokens: tokens)   # Wortlisten unverändert übernehmen
X_mini = cv_mini.fit_transform(mini)

pd.DataFrame(X_mini.toarray(), columns=cv_mini.get_feature_names_out())

,card,credit,error,fee,report
0,0,1,1,0,1
1,1,1,0,2,0
2,0,0,2,0,1


In [6]:
cv = CountVectorizer(analyzer=lambda tokens: tokens)
X_bow = cv.fit_transform(df["tokens_filtered"])

print("Form der Matrix:", X_bow.shape)
print("Vokabular:", len(cv.get_feature_names_out()))

Form der Matrix: (19807, 3571)
Vokabular: 3571


In [7]:
n_zellen = X_bow.shape[0] * X_bow.shape[1]
print("Einträge ungleich 0:", X_bow.nnz)
print(f"Anteil der Nullen: {1 - X_bow.nnz / n_zellen:.1%}")

Einträge ungleich 0: 789760
Anteil der Nullen: 98.9%


In [ ]:
#Matrix 19.807 × 3.571, 98,9 % Nullen
print(X_bow.shape, X_bow.sum())

(19807, 3571) 1240778


In [9]:
# TF-IDF Test

from sklearn.feature_extraction.text import TfidfVectorizer

tv_mini = TfidfVectorizer(analyzer=lambda tokens: tokens)
X_tfidf_mini = tv_mini.fit_transform(mini)

pd.DataFrame(X_tfidf_mini.toarray(), columns=tv_mini.get_feature_names_out()).round(2)

,card,credit,error,fee,report
0,0.00,0.58,0.58,0.00,0.58
1,0.42,0.32,0.00,0.85,0.00
2,0.00,0.00,0.89,0.00,0.45


In [10]:
tv = TfidfVectorizer(analyzer=lambda tokens: tokens)
X_tfidf = tv.fit_transform(df["tokens_filtered"])

print("Form der Matrix:", X_tfidf.shape)
print("Vokabular:", len(tv.get_feature_names_out()))
print("Einträge ungleich 0:", X_tfidf.nnz)

Form der Matrix: (19807, 3571)
Vokabular: 3571
Einträge ungleich 0: 789760


## Vektorisierung: Festlegungen

Die Wortlisten aus der Vorverarbeitung werden unverändert übernommen. Scikit-learn nimmt keine eigene Zerlegung, Kleinschreibung oder Filterung mehr vor.

Bag of Words und TF-IDF  nutzen damit dasselbe Vokabular (3.571 Wörter) und dieselben 19.807 Beschwerden. Sie unterscheiden sich nur in der Gewichtung der Einträge.

In [11]:
import numpy as np

words = cv.get_feature_names_out()
assert (words == tv.get_feature_names_out()).all()   # gleiche Reihenfolge in beiden Matrizen

bow_sum = np.asarray(X_bow.sum(axis=0)).ravel()       # Gesamthäufigkeit je Wort
tfidf_sum = np.asarray(X_tfidf.sum(axis=0)).ravel()   # Summe der TF-IDF-Gewichte je Wort

top_bow = pd.Series(bow_sum, index=words).nlargest(15)
top_tfidf = pd.Series(tfidf_sum, index=words).nlargest(15)

pd.DataFrame({
    "BoW (Häufigkeit)": top_bow.index,
    "TF-IDF (Summe Gewichte)": top_tfidf.index,
})

,BoW (Häufigkeit),TF-IDF (Summe Gewichte)
0,account,account
1,credit,credit
2,payment,report
3,report,payment
4,pay,debt
5,loan,pay
6,receive,loan
7,tell,information
8,time,receive
9,information,company
